# K-SPADA Binning Data Preparation

**Audience:** users reproducing the expression preprocessing used by the historical-best liver cancer experiment.  
**Goal:** preserve zero expression and map nonzero values to cell-wise quantile bins `1..7`, producing an integer matrix with range `0..7`.

This is scGPT-style quantile binning, not equal-width binning.

In [ ]:
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
import KSPADA

PROJECT_ROOT = Path('/home/lixiangyu/sh/program/K-SPADA')
DATA_DIR = Path('/home/lixiangyu/TOSICA-main/data/liver_cancer')
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'tutorial_binning'
SOURCE_H5AD = DATA_DIR / 'all_hvg_3000.h5ad'
OUTPUT_H5AD = OUTPUT_DIR / 'all_hvg_3000_scgpt_binned_0_7.h5ad'
BIN_COUNT = 8
RANDOM_SEED = 1


## Step 1 — Demonstrate zero-preserving quantile binning

In [ ]:
example_expression = np.array([
    [0.0, 0.2, 0.5, 1.0, 2.0, 4.0],
    [0.0, 0.0, 3.0, 3.0, 6.0, 9.0],
], dtype=np.float32)
example_binned = KSPADA.bin_expression_matrix(example_expression, bin_count=BIN_COUNT, random_seed=RANDOM_SEED)
display(pd.DataFrame(example_expression, index=['cell_1', 'cell_2']))
display(pd.DataFrame(example_binned, index=['cell_1', 'cell_2']))
assert np.all(example_binned[example_expression == 0] == 0)
assert example_binned.min() == 0
assert example_binned.max() <= BIN_COUNT - 1

## Step 2 — Apply binning to the liver cancer AnnData object

Load the complete expression matrix, apply zero-preserving quantile binning, and save the resulting AnnData object.


In [ ]:
source_adata = ad.read_h5ad(SOURCE_H5AD)
binned_adata = KSPADA.bin_adata(
    source_adata,
    bin_count=BIN_COUNT,
    random_seed=RANDOM_SEED,
    copy=True,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
binned_adata.write_h5ad(OUTPUT_H5AD)
print('Saved:', OUTPUT_H5AD)


## Step 3 — Audit the historical-best prepared file

In [ ]:
historical_binned_path = KSPADA.BEST_LIVER_CANCER_TUTORIAL.train_valid_path
historical_adata = ad.read_h5ad(historical_binned_path, backed='r')
row_chunk_size = 1024
chunk_ranges = range(0, historical_adata.n_obs, row_chunk_size)
historical_min = min(
    float(np.asarray(historical_adata.X[row_start:row_start + row_chunk_size]).min())
    for row_start in chunk_ranges
)
chunk_ranges = range(0, historical_adata.n_obs, row_chunk_size)
historical_max = max(
    float(np.asarray(historical_adata.X[row_start:row_start + row_chunk_size]).max())
    for row_start in chunk_ranges
)
display(pd.Series({
    'cells': historical_adata.n_obs,
    'genes': historical_adata.n_vars,
    'minimum_bin': historical_min,
    'maximum_bin': historical_max,
}))
historical_adata.file.close()
if (historical_min, historical_max) != (0.0, 7.0):
    raise ValueError('Historical training data must use bins 0 through 7.')

## Exercise

Explain why zero expression is kept as bin 0 instead of being mixed with the lowest nonzero quantile.

In [ ]:
exercise_answer = (
    'Bin 0 represents absence of detected expression, while bins 1-7 preserve the relative '
    'rank of nonzero expression within each cell.'
)
print(exercise_answer)

## Pitfall and extension

**Pitfall:** do not replace the quantile edges with equally spaced edges; that changes the historical preprocessing contract.  
**Extension:** compare the distribution of bins before and after HVG selection.